In [1]:
!pip install pyproj pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 11.4 MB/s  0:00:00m0:00:0100:01


In [2]:
import xml.etree.ElementTree as ET
import pandas as pd
import numpy as np

from pyproj import CRS, Transformer


# ============================================================
# CONFIGURACIÓN
# ============================================================

xml_file = "/mnt/c/Users/saulm/Documents/GitHub/theisNashEnvironmentalMonitoringModelingSite/l0rawData/Rst/ADCPMeasurements/1_0/20260807_172824_4420_QRev.xml"

output_file = "QRev_LatLonDepth.csv"


# ============================================================
# COORDENADA GEOGRÁFICA DE REFERENCIA
# ============================================================

# Coordenada del punto inicial de la transecta
lat0 = -15.123456
lon0 = -73.123456


# ============================================================
# DETERMINAR UTM AUTOMÁTICAMENTE
# ============================================================

utm_zone = int((lon0 + 180) / 6) + 1

if lat0 >= 0:
    epsg = 32600 + utm_zone
else:
    epsg = 32700 + utm_zone

print(f"UTM zone: {utm_zone}")
print(f"EPSG: {epsg}")


# ============================================================
# TRANSFORMADORES
# ============================================================

wgs84 = CRS.from_epsg(4326)
utm = CRS.from_epsg(epsg)

to_utm = Transformer.from_crs(
    wgs84,
    utm,
    always_xy=True
)

to_wgs84 = Transformer.from_crs(
    utm,
    wgs84,
    always_xy=True
)


# Coordenadas UTM del punto de referencia
x0, y0 = to_utm.transform(lon0, lat0)


# ============================================================
# LEER XML
# ============================================================

tree = ET.parse(xml_file)
root = tree.getroot()


# ============================================================
# EXTRAER MEASUREMENT POINTS
# ============================================================

measurement_points = root.findall(
    ".//CrossSectionSurvey/MeasurementPoint"
)

print(f"Measurement points: {len(measurement_points)}")


points = []


for point in measurement_points:

    def get_value(tag):

        element = point.find(tag)

        if element is None:
            return np.nan

        if element.text is None:
            return np.nan

        text = element.text.strip()

        if text.lower() == "nan" or text == "":
            return np.nan

        try:
            return float(text)

        except ValueError:
            return np.nan


    # -----------------------------------------
    # Distancias locales de QRev
    # -----------------------------------------

    distance_x = get_value("DistanceX")
    distance_y = get_value("DistanceY")

    depth = get_value("Depth")


    if not np.isfinite(distance_x):
        continue

    if not np.isfinite(distance_y):
        continue

    if not np.isfinite(depth):
        continue


    # ========================================================
    # CONVERTIR DISTANCIAS LOCALES A UTM
    # ========================================================

    utm_x = x0 + distance_x
    utm_y = y0 + distance_y


    # ========================================================
    # UTM -> LONGITUDE / LATITUDE
    # ========================================================

    lon, lat = to_wgs84.transform(
        utm_x,
        utm_y
    )


    points.append([
        lon,
        lat,
        depth
    ])


# ============================================================
# DATAFRAME
# ============================================================

df = pd.DataFrame(
    points,
    columns=[
        "Longitude",
        "Latitude",
        "Depth_m"
    ]
)


# ============================================================
# EXPORTAR
# ============================================================

df.to_csv(
    output_file,
    index=False,
    float_format="%.8f"
)


print("\n========================================")
print("EXPORT COMPLETED")
print("========================================")

print(f"Points: {len(df)}")
print(f"Output: {output_file}")

print("\nFirst points:")
print(df.head(10))

UTM zone: 18
EPSG: 32718
Measurement points: 71

EXPORT COMPLETED
Points: 71
Output: QRev_LatLonDepth.csv

First points:
   Longitude   Latitude  Depth_m
0 -73.123438 -15.123462    0.657
1 -73.123421 -15.123468    0.657
2 -73.123403 -15.123473    0.636
3 -73.123385 -15.123479    0.642
4 -73.123368 -15.123485    0.632
5 -73.123350 -15.123491    0.635
6 -73.123332 -15.123496    0.696
7 -73.123314 -15.123502    0.772
8 -73.123297 -15.123508    0.862
9 -73.123279 -15.123514    1.086
